<a href="https://colab.research.google.com/github/raya-lucaria/ia_o26/blob/main/course/9_aprendizaje_maquina/_assets/06_leer_el_error.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab" width="300"></a>

# Notebook 6 · Leer el error

Acompaña la página [Leer el error](https://raya-lucaria.github.io/ia_o26/aprendizaje-maquina/leer-el-error/leer-el-error/). Duración: ~25 min.

Ejecuta primero la celda de arranque y recorre el notebook de arriba abajo.
Antes de correr cada «Juega», anota tu predicción. Mueve el control indicado,
lee la gráfica y después abre «Explicación».

In [ ]:
#@title ▶ Corre esta celda primero
import os, sys, glob, base64, zlib, importlib, urllib.request
URL = os.environ.get("ML_LAB_URL",
      "https://raw.githubusercontent.com/raya-lucaria/ia_o26/main/cou"
      "rse/9_aprendizaje_maquina/_assets/ml_lab.py")
REQUIERE = "1.5"                     # la versión con que se generó este notebook
_RESPALDO = "..."

def _v(texto): return tuple(int(p) for p in texto.split("."))

def _sirve(version):                 # misma X y una Y mayor o igual
    (x, y), (x0, y0) = _v(version), _v(REQUIERE)
    return x == x0 and y >= y0

def _cargar():                       # importa ml_lab DE CERO (sin la copia en memoria)
    sys.modules.pop("ml_lab", None)
    for viejo in glob.glob("__pycache__/ml_lab.*.pyc"):
        os.remove(viejo)             # un .pyc de la copia vieja taparía la nueva
    importlib.invalidate_caches()
    import ml_lab
    return ml_lab if _sirve(ml_lab.VERSION) else None

ml = None
if os.path.exists("ml_lab.py"):      # 1) la copia local, si ya es nueva
    try: ml = _cargar()
    except Exception: ml = None
if ml is None:                       # 2) la del repositorio
    try:
        urllib.request.urlretrieve(URL, "ml_lab.py"); ml = _cargar()
    except Exception: ml = None
if ml is None:                       # 3) el respaldo: sin red o repo aún sin esta versión
    with open("ml_lab.py", "wb") as archivo:
        archivo.write(zlib.decompress(base64.b64decode(_RESPALDO)))
    ml = _cargar()
ml.requiere(REQUIERE)
ml.tema("claro")
from ml_lab import *
print(f"Listo: ml_lab {ml.VERSION}")

Lee el error antes de cambiar el modelo: su centro, ancho y forma cuentan
cosas distintas. El residual es $e_i=y_i-\widehat y_i$; $\bar e_{\mathcal D}$
es su promedio en un conjunto $\mathcal D$ de tamaño $n_{\mathcal D}$.
La identidad «Centro y ancho» de
[Leer el error](https://raya-lucaria.github.io/ia_o26/aprendizaje-maquina/leer-el-error/leer-el-error/)
separa la pérdida cuadrática promedio en centro al cuadrado y dispersión:

$$
L_{\mathcal D}=\bar e_{\mathcal D}^{\,2}+\frac1{n_{\mathcal D}}\sum_{i\in\mathcal D}(e_i-\bar e_{\mathcal D})^2.
$$

**Predice.** Con el modelo correcto, si duplicas el ruido $\sigma_\varepsilon$, ¿el histograma de residuales de validación se corre de lugar o solo se ensancha?

Elige: cambia el centro, cambia el ancho o cambian ambos sistemáticamente.
Deja «modelo» en «recta (el correcto)» y «dónde se leen» en «validación».
En «ruido σ_ε», pasa de 1 a 2 sin mover «semilla».

In [ ]:
#@title Juega: las tres vistas
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEMILLA = 0
F_B1 = {"correcto": lambda d: 1 + 2 * d.x1,
        "curva": lambda d: 1 + 2 * d.x1 + d.x1 ** 2,
        "omitida": lambda d: 1 + 2 * d.x1 + 1.5 * d.x2}
X_B1 = {"x1": ml.uniforme(-2, 2), "x2": ml.uniforme(-2, 2)}


def bloque_1(modelo, sigma, donde, semilla):
    datos = ml.proceso(n=300, x=X_B1, f=F_B1[modelo], ruido=ml.normal(0, sigma),
                       omitir=["x2"], semilla=semilla)
    m = ml.ajustar(datos, phi=ml.columnas("x1"))
    df = datos.val if donde == "validación" else datos.ent
    e = df["y"] - m.predecir(df)
    vista, hist = ml.dibujar_tres_paneles(m, df).axes[:2]   # ŷ contra y: mismos límites
    lim = vista.get_xlim() + vista.get_ylim(); vista.axis([min(lim), max(lim)] * 2)
    hist.axvspan(e.mean() - e.std(), e.mean() + e.std(), color="0.6", alpha=0.3)
    hist.axvline(0, ls="--", color="0.3"); hist.axvline(e.mean(), color="0.1")
    plt.show()
    media = ml.benchmarks(datos, df).loc["media", "cuadratica"]
    nombre = "val" if donde == "validación" else "ent"
    print(f"L_{nombre} = {ml.L(m, df):.3f}   ·   centro ē_{nombre} = {e.mean():.3f}"
          f"   ·   la media de entrenamiento (benchmark): {media:.3f}")
    print(f"R (despliegue, 100 000 obs.) = {ml.riesgo(m, datos):.3f}")


ml.laboratorio(
    bloque_1,
    modelo=ml.selector("modelo", {"recta (el correcto)": "correcto",
                                  "recta a una curva": "curva",
                                  "sin ver x2": "omitida"}, "correcto"),
    sigma=ml.deslizador("ruido σ_ε", 0.0, 3.0, 0.25, 1.0),
    donde=ml.selector("dónde se leen", ["validación", "entrenamiento"], "validación"),
    semilla=ml.deslizador("semilla", 0, 9, 1, SEMILLA),
)

**Mira.** Cada punto del primer panel compara predicción y observación; la diagonal indica igualdad. El histograma central cuenta residuales por intervalo.
Compara su ancho y el centro impreso; en el tercer panel, cada punto ubica un residual frente a $x_1$.
En el histograma, la línea punteada marca el 0, la continua el centro $\bar e$ y la banda sombreada va de una desviación por debajo a una por encima del centro.

<details><summary>Explicación</summary>

Se ensancha; en promedio, el centro no se desplaza. En una muestra de
validación sí puede moverse un poco. «Centro y ancho» separa ambos efectos;
«El centro no es el sesgo» de Leer el error distingue ese promedio del sesgo
de una predicción.

Si después mueves el control «semilla», la población y $f$ se quedan; cambian los datos, el ajuste y, con él, su riesgo $R(\widehat\beta)$.

Prueba «recta a una curva»: la curvatura omitida aparece como U en el tercer
panel. El intercepto compensa su promedio, pero no elimina la forma.
Con esa familia equivocada, el centro de validación puede variar incluso sin
ruido. Con el modelo correcto y sin ruido, la recta recupera $f$.

Al elegir «entrenamiento», el centro es cero, salvo precisión numérica:
la ecuación del intercepto obliga a que los residuales sumen cero.
Debajo se imprimen $L$ y el centro del conjunto elegido.
La media de entrenamiento es el punto de comparación (*benchmark*);
$R$ aproxima el riesgo cuadrático en el despliegue.

</details>

**Predice.** Si a todas las $y$ les sumas 1000, ¿cambia la mejora del modelo sobre el benchmark de la media?

Anota si aumenta, disminuye o queda igual. En «nivel: se suma a y», compara
0 con 1000, sin cambiar «fuerza de la señal».
Después fija la señal en 0.25 y activa
«dividir el RMSE entre el nivel ȳ (y no entre la dispersión de y)».

In [ ]:
#@title Juega: ¿bueno o malo?
SEMILLA = 0


def bloque_2(nivel, senal, entre_nivel):
    datos = ml.proceso(n=1000, x={"x": ml.uniforme(-1, 1)},
                       f=lambda d: nivel + senal * 2 * d.x, ruido=ml.normal(0, 1),
                       semilla=SEMILLA)
    m = ml.ajustar(datos)                       # se ajusta solo con entrenamiento
    ml.dibujar_benchmarks(m, datos)             # en validación
    plt.show()
    val, media = datos.val, datos.ent["y"].mean()
    l_mod = (val["y"] - m.predecir(val)) ** 2
    l_med = (val["y"] - media) ** 2
    rmse = np.sqrt(l_mod.mean())
    if entre_nivel:
        print(f"RMSE / ȳ = {rmse / media:.4f}   (dividir entre el nivel)")
    else:
        print(f"RMSE / desviación de y en ent = {rmse / datos.ent['y'].std():.3f}")
    print(f"mejora sobre la media (val) = {1 - l_mod.mean() / l_med.mean():.3f}")
    dif, ee = ml.diferencia_pareada(l_mod.to_numpy(), l_med.to_numpy())
    gana = "sí" if dif < -2 * ee else "no"
    print(f"diferencia pareada modelo − media en val = {dif:.3f} ± {ee:.3f} (1 EE)")
    print(f"¿le gana por más de 2 EE en validación, que aquí no eligió nada? {gana}")


ml.laboratorio(
    bloque_2,
    nivel=ml.deslizador("nivel: se suma a y", 0, 1000, 100, 0),
    senal=ml.deslizador("fuerza de la señal", 0.0, 2.0, 0.125, 1.0),
    entre_nivel=ml.interruptor("dividir el RMSE entre el nivel ȳ "
                               "(y no entre la dispersión de y)", False),
)

**Mira.** Cada barra horizontal muestra RMSE o MAE: compara modelo, media y mediana en validación con el piso poblacional correspondiente.
Con nivel 1000 y señal 0.25, contrasta el cociente impreso con la cercanía entre las barras del modelo y la media.
Al apagar el interruptor, el denominador es la desviación de $y$ en entrenamiento; debajo aparece la diferencia pareada con su EE.

<details><summary>Explicación</summary>

La mejora queda igual. El intercepto absorbe la suma: observaciones,
predicciones y media se trasladan juntas. Se conservan sus errores.
Aquí ajustas sin penalizar y con $\Phi$ de rango completo.
Dividir el RMSE entre el nivel puede hacerlo parecer pequeño sin mejorar
ninguna predicción. Dividir entre la dispersión compara escalas pertinentes.

«Benchmark» en Leer el error compara pérdidas por observación sobre las
mismas filas. La diferencia es modelo menos media: un valor negativo favorece
al modelo. Su error estándar, EE, mide la incertidumbre de ese promedio.
La celda informa si queda por debajo de cero por más de 2 EE.

Aquí validación sirve porque no eligió nada: ambos modelos se fijaron con
entrenamiento. Si eliges la señal o el modelo mirando esos resultados,
necesitas otra reserva para confirmar la mejora.
«La validación cruzada no es una reserva» impide trasladar ese EE a la
validación cruzada (*cross-validation*).

Con señal débil, la mejora sobre la media puede ser pequeña y el modelo
estar cerca del piso. «El piso: error de Bayes» explica ese límite poblacional
para la pérdida elegida. Una barra muestral puede quedar a cualquiera de
sus lados.

</details>

**Predice.** Con ruido de Pareto de exponente 1.5, si pasas de 100 a 10 000 observaciones de validación, ¿el MSE se estabiliza como con ruido normal?

Elige: se estabiliza o sigue dando saltos. Deja «dónde vive la cola» en
«en el ruido» y «qué se promedia» en «cuadrados (MSE)».
Compara «Pareto, exponente 1.5» con «normal» en «ruido».
Recorre el eje horizontal desde 100 hasta 10 000; no hay deslizador de tamaño.

La curva de la izquierda promedia el ruido de cada observación: es lo que
darían los residuales de validación de un modelo que acertara $f$ exactamente.

In [ ]:
#@title Juega: colas largas
SEMILLA = 0
RUIDOS_B3 = {"normal": ml.normal(), "t3": ml.t_student(3),
             "pareto": ml.pareto(1.5), "cauchy": ml.cauchy()}


def _datos_b3(ruido, donde, semilla):
    if donde == "x":     # la cola vive en x: x lognormal, f curva, ruido normal
        return ml.proceso(n=1000, x={"x": ml.lognormal(0, 1)}, semilla=semilla,
                          f=lambda d: 2 + np.exp(d.x / 3), ruido=ml.normal(0, 0.5))
    return ml.proceso(n=1000, x={"x": ml.uniforme(0, 1)}, f=lambda d: 2 + 3 * d.x,
                      ruido=RUIDOS_B3[ruido], semilla=semilla)


def bloque_3(ruido, que, donde, semilla):
    datos = _datos_b3(ruido, donde, semilla)
    m, val = ml.ajustar(datos), datos.val       # una recta en los dos casos
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.8))
    if donde == "x":                            # la vista 3: e contra x
        a1.scatter(val["x"], val["y"] - m.predecir(val), s=12)
        a1.set_xlabel("x (validación)"); a1.set_ylabel("residual e")
    else:
        ml.dibujar_media_acumulada({ruido: RUIDOS_B3[ruido]}, estadistico=que,
                                    semilla=semilla, ax=a1)
    ml.dibujar_peso_de_filas(m, val, ax=a2)
    plt.show()
    print("Los 5 peores errores en validación:")
    print(ml.peores_errores(m, val, k=5).round(2).to_string())


ml.laboratorio(
    bloque_3,
    ruido=ml.selector("ruido", {"normal": "normal", "t de Student, 3 g.l.": "t3",
                                "Pareto, exponente 1.5": "pareto", "Cauchy": "cauchy"},
                      "pareto"),
    que=ml.selector("qué se promedia", {"cuadrados (MSE)": "cuadrado",
                                        "valores absolutos (MAE)": "absoluto"}),
    donde=ml.selector("dónde vive la cola", {"en el ruido": "ruido",
                                             "en x (ruido normal)": "x"}),
    semilla=ml.deslizador("semilla", 0, 9, 1, SEMILLA),
)

**Mira.** A la izquierda, cada línea promedia ruido al cuadrado, o absoluto, para una semilla; compara las 20 trayectorias hacia la derecha.
A la derecha, las curvas acumulan fracciones del MSE y MAE de validación; ubica cuánto aporta el 1 % de filas con mayor error absoluto.
La tabla muestra los cinco peores errores de validación; con «en x (ruido normal)», cada punto izquierdo es un residual frente a $x$.

<details><summary>Explicación</summary>

No se estabiliza en un valor finito. Pareto de exponente 1.5 tiene segundo
momento infinito: los extremos provocan saltos del promedio cuadrático.
El piso cuadrático es infinito. «Colas largas» en Leer el error explica
por qué una muestra finita siempre entrega una cifra, aunque no haya un
riesgo finito que estimar.

Cambia «qué se promedia» a «valores absolutos (MAE)»: Pareto 1.5 tiene
primer momento absoluto finito y ese promedio sí converge.
Con «Cauchy», también el piso absoluto es infinito.
Un extremo de esta cola no es un outlier: pertenece al fenómeno.

Con «en x (ruido normal)», también cambia el proceso: $x$ es lognormal y
$f=2+e^{x/3}$, pero sigues ajustando una recta. Conservas tamaño y semilla.
Los errores grandes se concentran donde $x$ es raro y la recta extrapola
mal la curva. Con cola en el ruido pueden aparecer en cualquier $x$.

</details>

**Predice.** Si el modelo no ve $x_2$, y $\operatorname{corr}(x_1,x_2)=0.3$, ¿dónde se centra el histograma de $\widehat\beta_1$ en 200 entrenamientos: en 2, en 2.9 o en 5?

El proceso es $y=1+2x_1+3x_2+\varepsilon$, con variables de varianza 1.
Elige una opción. Deja «qué no ve el modelo» en «x2», «corr(x1, x2)»
en 0.3 y «corr(x1, ε): endogeneidad» en 0.
Apaga «agregar la columna que falta». Una combinación nueva puede tardar
unos segundos.

In [ ]:
#@title Juega: lo que el modelo no ve
SEMILLA, _CACHE_B4 = 0, {}

def _datos_b4(que, corr, endog, agregar, semilla, realidad=True):
    prod = 2 if que == "producto" else 0        # f = 1 + 2·x1 + 3·x2 (+ 2·x1·x2)
    return ml.proceso(n=200, x={"x1": ml.normal(), "x2": ml.normal()},
                      f=lambda d: 1 + 2 * d.x1 + 3 * d.x2 + prod * d.x1 * d.x2,
                      ruido=ml.normal(0, 1), endogeno=endog,
                      correlacion={("x1", "x2"): corr} if corr else None,
                      omitir=["x2"] if que == "x2" and not agregar else None,
                      semilla=semilla, con_realidad=realidad)

def bloque_4(que, corr, endog, agregar):
    phi = (ml.combinar(ml.columnas("x1", "x2"), ml.interacciones(("x1", "x2")))
           if que == "producto" and agregar else None)   # None: todo lo visible
    clave = (que, corr, endog, agregar)
    if clave not in _CACHE_B4:                  # 200 entrenamientos, una vez
        fab = lambda s: _datos_b4(que, corr, endog, agregar, s, realidad=False)
        _CACHE_B4[clave] = ml.beta_repetido(fab, phi, 200)
    cuenta = 2 + (3 * corr if que == "x2" and not agregar else 0) + endog
    ml.dibujar_beta_repetido(_CACHE_B4[clave], verdadero={"x1": 2.0}, columnas=("x1",))
    plt.axvline(cuenta, color="gray", ls="--", label="la cuenta")
    plt.legend(); plt.show()
    datos = _datos_b4(que, corr, endog, agregar, SEMILLA)
    m = ml.ajustar(datos, phi=phi)
    print(f"media de 200 β̂₁ = {_CACHE_B4[clave]['x1'].mean():.3f}"
          f" · la cuenta = {cuenta:.2f}\nL_val = {ml.L(m, datos.val):.3f}"
          f" · piso con lo visible = {ml.piso(datos):.3f}")
    if que == "producto":
        ml.dibujar_residuales_plano(m, datos.val, "x1", "x2"); plt.show()
        ml.imprimir_Phi(m, datos.val, filas=3)

ml.laboratorio(
    bloque_4,
    que=ml.selector("qué no ve el modelo", {"nada": "nada", "x2": "x2",
                                            "el producto x1·x2": "producto"}, "x2"),
    corr=ml.deslizador("corr(x1, x2)", 0.0, 0.9, 0.1, 0.3),
    endog=ml.deslizador("corr(x1, ε): endogeneidad", 0.0, 0.6, 0.1, 0.0),
    agregar=ml.interruptor("agregar la columna que falta", False),
)

**Mira.** Cada entrenamiento aporta una pendiente al histograma; compara su centro con la línea verde del valor verdadero y la gris de la cuenta.
Con «el producto x1·x2», cada marca del plano es un residual: orientación y color indican signo; tamaño indica magnitud.
Activa «agregar la columna que falta»: compara el plano y las columnas de $\Phi$ antes y después, además de $L_{\text{val}}$ y el piso impresos.

<details><summary>Explicación</summary>

Se centra en $2+3\cdot0.3=2.9$. La recta atribuye a $x_1$ parte del efecto
de $x_2$. «Variable omitida» en Leer el error explica la cuenta.
Con varianzas unitarias, para este ajuste que solo ve $x_1$, se reduce a:

$$
2+3\,\operatorname{corr}(x_1,x_2)+\operatorname{corr}(x_1,\varepsilon).
$$

El segundo término recoge la variable omitida; el tercero, la endogeneidad.
Con ambos controles de correlación en cero, la cuenta vuelve a 2.
Las variables normales son entonces independientes: omitir $x_2$ solo sube
el piso cuadrático.

Pon «qué no ve el modelo» en «nada» y aumenta «corr(x1, ε): endogeneidad».
El centro puede desplazarse sin omitir variables.
«Endogeneidad» explica por qué los residuales no revelan ese sesgo del
estimador del coeficiente estructural. El mejor predictor cuadrático busca
$E[y\mid x_{\text{visibles}}]$, que aquí no coincide con $f$.

Combina «x2», «corr(x1, x2)» en 0 y endogeneidad en 0.4.
El piso impreso viene de `ml.piso(datos)` e incluye la variable omitida.
No uses el piso de la endogeneidad sola. Ese límite corresponde al riesgo
poblacional, no obliga a cada $L_{\text{val}}$ muestral a quedar encima.

Con «el producto x1·x2», ambas variables son visibles; falta su interacción
en $\Phi$. Agregar el producto incorpora una columna: el modelo sigue
siendo lineal en sus parámetros.

</details>

**Predice.** Con $\operatorname{corr}(x_1,x_2)=0.99$, ¿qué varía más entre 200 entrenamientos: $\widehat\beta_1$ o la predicción en $x_1=x_2=1$?

Elige entre la pendiente y la predicción. Pon «corr(x1, x2)» en 0.99 y
«dónde predices» en «dentro de la nube: x1 = x2 = 1».
Cada correlación nueva calcula los entrenamientos una vez; espera unos
segundos antes de comparar las desviaciones impresas.

In [ ]:
#@title Juega: reaprender con colinealidad
SEMILLA = 0
CORRS_B5 = [0.0, 0.5, 0.9, 0.99, 0.999]
PUNTOS_B5 = {"dentro": (1.0, 1.0), "fuera": (1.0, -1.0)}


def _fab_b5(corr):
    return lambda s: ml.caso(9, semilla=s, correlacion={("x1", "x2"): corr},
                             con_realidad=False)


TABLAS_B5 = {}                                  # cada (correlación, semilla), una vez


def bloque_5(corr, donde, semilla):
    if (corr, semilla) not in TABLAS_B5:
        TABLAS_B5[corr, semilla] = ml.beta_repetido(_fab_b5(corr), None, 200,
                                                    semilla=semilla)
    tabla = TABLAS_B5[corr, semilla]
    x1, x2 = PUNTOS_B5[donde]
    y_gorro = tabla["1"] + tabla["x1"] * x1 + tabla["x2"] * x2
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
    ml.dibujar_beta_repetido(tabla, verdadero={"x1": 1.0, "x2": 1.0},
                             columnas=("x1", "x2"), ax=a1)
    a2.hist(y_gorro, bins=30)
    a2.set_xlabel(f"ŷ en x1 = {x1:g}, x2 = {x2:g}")
    plt.show()
    print(f"desviación de β̂₁ en 200 entrenamientos = {tabla['x1'].std():.3f}")
    print(f"desviación de ŷ en ese punto = {y_gorro.std():.3f}")


ml.laboratorio(
    bloque_5,
    corr=ml.selector("corr(x1, x2)", CORRS_B5, 0.9),
    donde=ml.selector("dónde predices",
                      {"dentro de la nube: x1 = x2 = 1": "dentro",
                       "fuera de la diagonal de los datos: x1 = 1, x2 = −1": "fuera"},
                      "dentro"),
    semilla=ml.deslizador("semilla", 0, 9, 1, SEMILLA),
)

**Mira.** Cada punto izquierdo reúne las dos pendientes de un entrenamiento; la nube se alarga donde su suma cambia poco.
El histograma derecho cuenta las predicciones de esos mismos entrenamientos en el punto elegido; compara su ancho con las desviaciones impresas.
En «dónde predices», cambia a «fuera de la diagonal de los datos: x1 = 1, x2 = −1» y observa cómo se ensancha.

<details><summary>Explicación</summary>

Varía más $\widehat\beta_1$. Con entradas muy parecidas, distintas pendientes
se compensan: la suma $\widehat\beta_1+\widehat\beta_2$ queda bien determinada.
«Colinealidad» en Leer el error muestra que la varianza de cada pendiente
crece con el factor $1/(1-\operatorname{corr}(x_1,x_2)^2)$.

En $x_1=x_2=1$, la predicción usa esa suma y el intercepto.
En $x_1=1,x_2=-1$, usa la diferencia de pendientes, que es inestable.
Predecir fuera de la diagonal extrapola hacia una región con pocos datos.
Ante esa inestabilidad, ¿podemos preferir una solución menos extrema?
El siguiente notebook explora esa posibilidad penalizando pendientes grandes.

</details>

**Taller.** Corre el proceso activo y cambia una sola línea por una alternativa
comentada. Conserva `ml.taller(datos, clase=3, que="error")`.
Para usar la interacción, cambia `f` y quita `omitir`.
Para los ruidos alternativos, sustituye el argumento `ruido`.

Compara centro, ancho, forma y errores frente al benchmark y al piso.
Después revisa las peores filas y las pendientes al repetir entrenamientos.
Anota qué cambió y qué evidencia gráfica lo muestra.
Con `ruido=ml.t_student(2)`, ignora «a más de 2 EE»: sin varianza finita de la pérdida, ese EE no mide incertidumbre (la definición del punto de comparación de la página exige $E[\ell^2]<\infty$).

In [ ]:
# Tu proceso: cambia una línea y vuelve a correr.
datos = ml.proceso(n=300,
                   x={"x1": ml.normal(), "x2": ml.normal()},
                   f=lambda d: 1 + 2 * d.x1 + 3 * d.x2,
                   ruido=ml.normal(0, 1),
                   correlacion={("x1", "x2"): 0.5},
                   omitir=["x2"], semilla=0)
ml.taller(datos, clase=3, que="error")
# 1) ruido=ml.heterocedastico(lambda d: 0.5 + abs(d.x1))
# 2) f=lambda d: 1 + 2 * d.x1 + 3 * d.x1 * d.x2   y quita omitir
# 3) ruido=ml.t_student(2)

**Galería.** Pulsa un botón, lee su pregunta y responde antes de abrir
«¿Qué se ve?». Sustenta tu respuesta con una vista del error o con la
distribución de pendientes. Repite con otro botón y distingue qué mecanismo
produce el error en cada caso.
En el caso 6, ignora «a más de 2 EE»: sin varianza finita de la pérdida, ese EE no mide incertidumbre (la definición del punto de comparación de la página exige $E[\ell^2]<\infty$).

In [ ]:
#@title Galería: casos que se rompen a la vista
ml.galeria(clase=3, casos=[6, 7, 8, 9, 11, 12], que="error")

**Ejercicio.** Con endogeneidad 0.4, calcula en una línea la correlación entre
$e$ y $x_1$ en entrenamiento y en validación.
¿Por qué los residuales no delatan el problema?

En el taller, quita `omitir`, fija la correlación entre variables en 0 y
agrega `endogeno=0.4` a `ml.proceso`. Después ejecuta `m = ml.ajustar(datos)`.
Usa `datos.ent` y `datos.val`, con $e=y-\widehat y$.

**Pista:** ¿qué ecuación cumple el producto entre cada columna de $\Phi$
y los residuales de entrenamiento?

<details><summary>Respuesta</summary>

```python
[(df["y"] - m.predecir(df)).corr(df["x1"]) for df in (datos.ent, datos.val)]
```

La correlación de entrenamiento es cero, salvo precisión numérica, por las
ecuaciones normales. La de validación fluctúa alrededor de cero.
El ajuste absorbe la parte predecible del ruido correlacionado con $x_1$.
Por eso una nube sin patrón puede acompañar un estimador sesgado del
coeficiente estructural: estimar ese coeficiente y predecir son problemas
distintos.

</details>

**Ejercicio.** Compara dos procesos que ven las mismas variables $x_1$ y $x_2$.
Ajusta ambos con la misma $\Phi$: constante, $x_1$, $x_2$.
En A, la desviación del ruido crece con $\lvert x_1\rvert$.
En B, falta representar el producto $x_1x_2$.
Ambos abren un abanico en el residual contra $x_1$.
Encuentra una vista que los distinga.

Copia estos procesos en el taller, después de haber corrido el primer bloque:

```python
A = ml.proceso(n=600, x={"x1": ml.normal(), "x2": ml.normal()}, f=lambda d: 1 + d.x1 + d.x2,
               ruido=ml.heterocedastico(lambda d: np.sqrt(0.25 + d.x1 ** 2)), semilla=0)
B = ml.proceso(n=600, x={"x1": ml.normal(), "x2": ml.normal()},
               f=lambda d: 1 + d.x1 + d.x2 + d.x1 * d.x2, ruido=ml.normal(0, 0.5), semilla=0)
```

Ejecuta `ml.taller(A, clase=3, que="error")` y luego la misma llamada con `B`.
Si necesitas otra vista, `ml.dibujar_residuales_plano(m, datos.val, "x1", "x2")`
acepta el modelo `m = ml.ajustar(datos)`, con `datos = A` o `datos = B`.

**Pista:** ¿qué información pierdes al poner solo $x_1$ en el eje horizontal?

<details><summary>Respuesta</summary>

El plano $(x_1,x_2)$ conserva la variable que el abanico oculta.
En B, el signo del residual tiende a seguir al de $x_1x_2$:
aparece un patrón por cuadrantes. En A, los signos se mezclan;
crece la dispersión, pero no hay ese patrón en la media.

Ambos tienen la misma varianza condicional del error ideal dado $x_1$,
$0.25+x_1^2$. Los ajustes finitos pueden alterar algo los abanicos.
La página Leer el error pide completar el argumento; aquí ya tienes
la vista que permite distinguirlos.

</details>

Lee centro, ancho y forma antes de resumir el error.
Compara con un benchmark y con el piso de la pérdida que usas.
Revisa qué filas dominan el promedio y qué variables o columnas faltan.
Repetir entrenamientos distingue pendientes inestables de predicciones
inestables.

El notebook 7 abre esta pregunta: ¿podemos preferir una solución menos extrema?